Markdown:

#***Data Augmentation***

This notebook performs data augmentation on the minority classes
(Biological and Ambience) to reduce class imbalance in the underwater
acoustic dataset.

Augmentations applied:

• Gaussian Noise

• Time Shift

• Pitch Shift

• Time Stretch

• Gain Adjustment

Install Dependencies

In [ ]:
!pip install -q librosa soundfile audiomentations tqdm

Import Libraries

In [ ]:
import os
import shutil
import random

import librosa
import numpy as np
import soundfile as sf

from tqdm.notebook import tqdm
from google.colab import drive

import matplotlib.pyplot as plt

Mount Google Drive

In [ ]:
drive.mount('/content/drive')

Define Project Paths

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/Underwater Audio Data"

PROCESSED_DIR = os.path.join(PROJECT_DIR, "processed")
AUGMENTED_DIR = os.path.join(PROJECT_DIR, "augmented")

Verify Folder Structure

In [ ]:
print("Processed Directory:")
print(PROCESSED_DIR)

print("\nClasses Found:")

for cls in os.listdir(PROCESSED_DIR):
    path = os.path.join(PROCESSED_DIR, cls)

    if os.path.isdir(path):
        print(f"{cls}: {len(os.listdir(path))} files")

Create Output Folder

In [ ]:
# Remove previous augmented dataset if it exists

if os.path.exists(AUGMENTED_DIR):
    shutil.rmtree(AUGMENTED_DIR)

os.makedirs(AUGMENTED_DIR)

for cls in os.listdir(PROCESSED_DIR):

    os.makedirs(
        os.path.join(AUGMENTED_DIR, cls),
        exist_ok=True
    )

print("Augmented folder created successfully.")

Count Original Files

In [ ]:
original_counts = {}

for cls in os.listdir(PROCESSED_DIR):

    class_path = os.path.join(PROCESSED_DIR, cls)

    if os.path.isdir(class_path):

        original_counts[cls] = len([
            f for f in os.listdir(class_path)
            if f.endswith(".wav")
        ])

print(original_counts)

Visualize Class Distribution

In [ ]:
plt.figure(figsize=(6,4))

plt.bar(
    original_counts.keys(),
    original_counts.values()
)

plt.title("Original Dataset Distribution")

plt.xlabel("Class")

plt.ylabel("Number of Audio Files")

plt.show()

Augmentation Functions

In [ ]:
# Gaussian Noise

def add_noise(y, noise_factor=0.005):

    noise = np.random.randn(len(y))

    augmented = y + noise_factor * noise

    return augmented.astype(np.float32)

# Time Shift

def time_shift(y, shift_max=0.2):

    shift = int(
        np.random.uniform(-shift_max, shift_max) * len(y)
    )

    return np.roll(y, shift)

# Pitch Shift

def pitch_shift(y, sr, steps=None):

    if steps is None:
        steps = random.uniform(-2, 2)

    return librosa.effects.pitch_shift(
        y,
        sr=sr,
        n_steps=steps
    )

# Time Stretch

def time_stretch(y):

    rate = random.uniform(0.9, 1.1)

    return librosa.effects.time_stretch(
        y,
        rate=rate
    )

# Gain

def gain(y):

    factor = random.uniform(0.8, 1.2)

    return y * factor

Copy Vessel Files

In [ ]:
print("Copying Vessel files...")

vessel_src = os.path.join(PROCESSED_DIR, "Vessels")
vessel_dst = os.path.join(AUGMENTED_DIR, "Vessels")

count = 0

for file in tqdm(os.listdir(vessel_src)):

    if file.endswith(".wav"):

        shutil.copy2(
            os.path.join(vessel_src, file),
            os.path.join(vessel_dst, file)
        )

        count += 1

print(f"Copied {count} vessel files.")

Augment Biological Files

We'll generate 3 augmented versions for each Biological recording.

In [ ]:
print("Augmenting Biological files...")

bio_src = os.path.join(PROCESSED_DIR, "Biological")
bio_dst = os.path.join(AUGMENTED_DIR, "Biological")

count = 0

for file in tqdm(os.listdir(bio_src)):

    if not file.endswith(".wav"):
        continue

    path = os.path.join(bio_src, file)

    y, sr = librosa.load(path, sr=None)

    base = os.path.splitext(file)[0]

    # Save original
    sf.write(
        os.path.join(bio_dst, file),
        y,
        sr
    )

    # Noise
    sf.write(
        os.path.join(bio_dst, base + "_noise.wav"),
        add_noise(y),
        sr
    )

    # Pitch Shift
    sf.write(
        os.path.join(bio_dst, base + "_pitch.wav"),
        pitch_shift(y, sr),
        sr
    )

    # Time Shift
    sf.write(
        os.path.join(bio_dst, base + "_shift.wav"),
        time_shift(y),
        sr
    )

    count += 4

print(f"Generated approximately {count} Biological files.")

Augment Ambience Files


We'll generate 5 augmented versions for each Ambience recording


In [ ]:
print("Augmenting Ambience files...")

amb_src = os.path.join(PROCESSED_DIR, "Ambience")
amb_dst = os.path.join(AUGMENTED_DIR, "Ambience")

count = 0

for file in tqdm(os.listdir(amb_src)):

    if not file.endswith(".wav"):
        continue

    path = os.path.join(amb_src, file)

    y, sr = librosa.load(path, sr=None)

    base = os.path.splitext(file)[0]

    # Save original
    sf.write(
        os.path.join(amb_dst, file),
        y,
        sr
    )

    # Noise
    sf.write(
        os.path.join(amb_dst, base + "_noise.wav"),
        add_noise(y),
        sr
    )

    # Pitch
    sf.write(
        os.path.join(amb_dst, base + "_pitch.wav"),
        pitch_shift(y, sr),
        sr
    )

    # Shift
    sf.write(
        os.path.join(amb_dst, base + "_shift.wav"),
        time_shift(y),
        sr
    )

    # Stretch
    sf.write(
        os.path.join(amb_dst, base + "_stretch.wav"),
        time_stretch(y),
        sr
    )

    # Gain
    sf.write(
        os.path.join(amb_dst, base + "_gain.wav"),
        gain(y),
        sr
    )

    count += 6

print(f"Generated approximately {count} Ambience files.")

Verify Final Dataset

In [ ]:
augmented_counts = {}

for cls in os.listdir(AUGMENTED_DIR):

    class_path = os.path.join(AUGMENTED_DIR, cls)

    augmented_counts[cls] = len([
        f for f in os.listdir(class_path)
        if f.endswith(".wav")
    ])

print("Augmented Dataset")

for k, v in augmented_counts.items():

    print(f"{k}: {v}")

Compare Before and After

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))

classes = list(original_counts.keys())

x = np.arange(len(classes))
width = 0.35

ax.bar(
    x - width/2,
    [original_counts[c] for c in classes],
    width,
    label="Original"
)

ax.bar(
    x + width/2,
    [augmented_counts[c] for c in classes],
    width,
    label="Augmented"
)

ax.set_xticks(x)
ax.set_xticklabels(classes)

ax.set_ylabel("Number of Audio Files")

ax.set_title("Dataset Before vs After Augmentation")

ax.legend()

plt.show()